In [1]:
import pandas as pd
import pickle
import argparse
import os
import xgboost as xgb
from nltk.sentiment.vader import SentimentIntensityAnalyzer
import nltk
from sklearn.preprocessing import MinMaxScaler

In [2]:


def main(input_path, output_path):
    """
    Main function to load datasets, process them, and make predictions using XGBoost model.

    Parameters:
    input_path (str): Path to the directory containing the input CSV files.
    output_path (str): Path to save the output CSV file with predictions.
    """
    # Load datasets
    df_stock = pd.read_csv(os.path.join(input_path, 'InputData\\stocks_prices_and_volumes.csv'))
    housing_index = pd.read_csv(os.path.join(input_path, 'InputData\\housing_index.csv'))
    oil = pd.read_csv(os.path.join(input_path, 'InputData\\crude_oil_prices.csv'))
    federal_fund = pd.read_csv(os.path.join(input_path, 'InputData\\effective_federal_funds_rate.csv'))
    df_gold_intraday = pd.read_csv(os.path.join(input_path, 'InputData\\intraday_gold.csv'))
    df_news = pd.read_excel(os.path.join(input_path, 'InputData\\news.xlsx'))
    mm_inflation = pd.read_csv(os.path.join(input_path, 'InputData\\inflation_month_on_month.csv'))
    yy_inflation = pd.read_csv(os.path.join(input_path, 'InputData\\inflation_year_on_year.csv'))
    corridor_interest = pd.read_csv(os.path.join(input_path, 'InputData\\egyptian_corridor_interest_rates.csv'))
    vix = pd.read_csv(os.path.join(input_path, 'InputData\\vix_index.csv'))
    vxeem = pd.read_csv(os.path.join(input_path, 'InputData\\vxeem_index.csv'))
    federal_fund  = pd.read_csv(os.path.join(input_path, 'InputData\\effective_federal_funds_rate.csv'))
    house_index = pd.read_csv(os.path.join(input_path, 'InputData\\housing_index.csv'))
    oil = pd.read_csv(os.path.join(input_path, 'InputData\\crude_oil_prices.csv'))
    
    
    
    
    #create final dataframe 
    final = pd.DataFrame()

  
    #preprocessing on stocks and volume file 
    #group all stocks in categories 
    # 1. Food, Beverages, and Tobacco
    df_stock['food_beverages_tobacco_avg'] = df_stock[['stock_0_food_beverages_and_tobacco_close_price']].mean(axis=1)
    # 2. Non-Banking Financial Services
    df_stock['non_banking_financial_avg'] = df_stock[['stock_9_non-banking_financial_services_close_price',
                                                   'stock_10_non-banking_financial_services_close_price',
                                                   'stock_12_non-banking_financial_services_close_price']].mean(axis=1)
    # 3. Real Estate
    df_stock['real_estate_avg'] = df_stock[['stock_6_real_estate_close_price', 'stock_7_real_estate_close_price',
                                         'stock_8_real_estate_close_price', 'stock_11_real_estate_close_price']].mean(axis=1)
    # 5. Banks
    df_stock['banks_avg'] = df_stock[['stock_4_banks_close_price', 'stock_5_banks_close_price']].mean(axis=1)
    # 6. Basic Resources
    df_stock['basic_resources_avg'] = df_stock[['stock_2_basic_resources_close_price', 'stock_3_basic_resources_close_price']].mean(axis=1)
    # 7. Energy and Support Services
    df_stock['energy_support_services_avg'] = df_stock[['stock_1_energy_and_support_services_close_price']].mean(axis=1)

    final['food_beverages_tobacco_return'] = df_stock['food_beverages_tobacco_avg'].pct_change()
    final['non_banking_financial_return'] = df_stock['non_banking_financial_avg'].pct_change()
    final['real_estate_return'] = df_stock['real_estate_avg'].pct_change()
    final['banks_return'] = df_stock['banks_avg'].pct_change()
    final['basic_resources_return'] = df_stock['basic_resources_avg'].pct_change()
    final['energy_support_services_return'] = df_stock['energy_support_services_avg'].pct_change()
    final['food_beverages_tobacco_volume'] = df_stock[['stock_0_food_beverages_and_tobacco_volume']].mean(axis=1)
    final['non_banking_financial_volume'] = df_stock[['stock_9_non-banking_financial_services_volume',
                                                      'stock_10_non-banking_financial_services_volume',
                                                      'stock_12_non-banking_financial_services_volume']].mean(axis=1)
    final['real_estate_volume'] = df_stock[['stock_6_real_estate_volume', 'stock_7_real_estate_volume',
                                            'stock_8_real_estate_volume', 'stock_11_real_estate_volume']].mean(axis=1)
    final['banks_volume'] = df_stock[['stock_4_banks_volume', 'stock_5_banks_volume']].mean(axis=1)
    final['basic_resources_volume'] = df_stock[['stock_2_basic_resources_volume', 'stock_3_basic_resources_volume']].mean(axis=1)
    final['energy_support_services_volume'] = df_stock[['stock_1_energy_and_support_services_volume']].mean(axis=1)
    final['market_proxy'] = final[['food_beverages_tobacco_return', 'non_banking_financial_return', 
                                   'real_estate_return', 'banks_return', 
                                   'basic_resources_return', 'energy_support_services_return']].mean(axis=1)
    final['market_proxy_trend']= final['market_proxy'].rolling(window=7).mean()
    
    
    
    
    #intraday preprocessing
    
    # Convert the timestamp column (replace 'timestamp' with your actual timestamp column name)
    df_gold_intraday['timestamp'] = pd.to_datetime(df_gold_intraday['Timestamp'], errors='coerce')
    df_gold_intraday['timestamp'] = df_gold_intraday['timestamp'].fillna(pd.to_datetime(df_gold_intraday['Timestamp'] + '+00:00', errors='coerce'))
    df_gold_intraday['date'] = df_gold_intraday['timestamp'].dt.date
    df_gold_daily = df_gold_intraday.groupby('date').agg({
        '24K': ['first', 'max', 'min', 'last']  # Open (first), High (max), Low (min), Close (last)
    })

    # Step 2: Flatten multi-level columns (from aggregation)
    df_gold_daily.columns = ['_'.join(col).strip() for col in df_gold_daily.columns]
    # Step 3: Rename columns for clarity
    df_gold_daily.rename(columns={
        '24K_first': 'open',   # First price (Open)
        '24K_max': 'high',     # Max price (High)
        '24K_min': 'low',      # Min price (Low)
        '24K_last': 'close_gold'    # Last price (Close)
    }, inplace=True)
    df_gold_daily['momentum_gold'] = df_gold_daily['close_gold'] - df_gold_daily['open']
    # 2. High-Low Difference percentage (Volatility)
    df_gold_daily['high_low_gold'] = ((df_gold_daily['high'] - df_gold_daily['low']) / df_gold_daily['low']) * 100
    # 5. Daily Percent Change
    df_gold_daily['daily_percent_change_gold'] = df_gold_daily['close_gold'].pct_change() * 100
    # Reset the index of both DataFrames to ensure alignment
    final.reset_index(drop=True, inplace=True)
    df_gold_daily.reset_index( inplace=True)
    # Now, combine them based on index (row order) and other features 
    final = pd.concat([ df_gold_daily[['date','momentum_gold', 'high_low_gold', 'close_gold', 'daily_percent_change_gold']],final], axis=1)
    final['gold_voltality_7'] = df_gold_daily['close_gold'].rolling(window=7).std()
    final['gold_voltality_30'] = df_gold_daily['close_gold'].rolling(window=30).std()
    final['daily_percent_change_gold_trend'] = final['daily_percent_change_gold'].rolling(window=7).mean()

    
    
    
    #news file preprocessing 
    # Convert the 'date' column to traditional date format (YYYY-MM-DD)
    df_news['date'] = pd.to_datetime(df_news['date'], errors='coerce')
    # Optional: Format the date to display in 'YYYY-MM-DD' format if needed
    df_news['date'] = df_news['date'].dt.strftime('%Y-%m-%d')
    # Convert the 'category' and 'tag' columns to lowercase before one-hot encoding
    df_news['tag'] = df_news['tag'].str.lower()
    # Perform one-hot encoding on 'category' and 'tag' columns
    df_news = pd.get_dummies(df_news, columns=['tag'], prefix=[ 'tag'], drop_first=False)
    # Download the VADER lexicon
    nltk.download('vader_lexicon')
    # Initialize the VADER sentiment intensity analyzer
    sia = SentimentIntensityAnalyzer()
    # Apply VADER sentiment analysis to 'translated_title' and 'summary'
    df_news['news_title'] = df_news['news_title'].apply(lambda title: sia.polarity_scores(str(title))['compound'])
    df_news['news_summary'] = df_news['news_summary'].apply(lambda summary: sia.polarity_scores(str(summary))['compound'])
    # Group by 'date' and sum the one-hot encoded columns as well as sentiment scores
    df_news_grouped = df_news.groupby('date').sum().reset_index()
    # Step 1: Ensure 'date' columns in both tables are in the same format (if needed)
    final['date'] = pd.to_datetime(final['date'])
    df_news_grouped['date'] = pd.to_datetime(df_news_grouped['date'])  # Assuming your news DataFrame is named `news_table`
    # Step 2: Create a full list of dates from the final table
    all_dates = pd.DataFrame(final['date'].unique(), columns=['date']).sort_values(by='date')
    # Step 3: Merge the full list of dates with the news table, filling missing rows with zeros
    news_table_full = pd.merge(all_dates, df_news_grouped, on='date', how='left').fillna(0)
        # Ensure 'date' columns in both tables are in the same format
    news_table_full['date'] = pd.to_datetime(news_table_full['date'])
    final['date'] = pd.to_datetime(final['date'])
    # Merge the two DataFrames based on the 'date' column
    final = pd.merge(final, news_table_full, on='date', how='left')
    
    
    
    #month on month and year on year inflation 
    final['inflation_month_headline']= mm_inflation['Headline ']
    final['inflation_month_core']= mm_inflation['Core ']
    final['inflation_year_headline']= yy_inflation['Headline']
    final['inflation_year_Core']= yy_inflation['Core']

    
    
    #corridor interest 
    final['Overnight Deposit Rate']= corridor_interest['Overnight Deposit Rate']
    final['Overnight Lending Rate']= corridor_interest['Overnight Lending Rate']
    final['std change Deposit Rate 30']= corridor_interest['Overnight Deposit Rate'].rolling(window=30).std()
    final['std change Lending Rate 30']= corridor_interest['Overnight Lending Rate'].rolling(window=30).std()
    
    #vix and vxeem index 
    final['vix_close']= vix['CLOSE']

    final['vix_close_pctChange']= vix['CLOSE'].pct_change()

    final['vix_range']= vix['HIGH'] - vix['LOW']

    final['vix_avg_7']= vix['CLOSE'].rolling(window=7).mean()
    
    final['vxeem_close']= vxeem['CLOSE']

    final['vxeem_close_pctChange']= vxeem['CLOSE'].pct_change()

    final['vxeem_range']= vxeem['HIGH'] - vxeem['LOW']

    final['vxeem_avg_7']= vxeem['CLOSE'].rolling(window=7).mean()
    
    #effr
    # Step 1: Convert the EFFR column to numeric, forcing any errors to NaN
    federal_fund['EFFR'] = pd.to_numeric(federal_fund['EFFR'], errors='coerce')

    # Step 2: Assign the EFFR column to the final DataFrame
    final['EFFR'] = federal_fund['EFFR']

    # Step 3: Calculate the percentage change for EFFR
    final['EFFR_pctChange'] = final['EFFR'].pct_change()

    final['EFFR_std 30'] = final['EFFR'].rolling(window=30).std()
    
    #house index 

    # Step 2: Assign the EFFR column to the final DataFrame
    final['housing_index'] = house_index['CSUSHPINSA']

    # Step 3: Calculate the percentage change for EFFR
    final['house_pct_change'] = house_index['CSUSHPINSA'].pct_change()

    final['house_volt'] = house_index['CSUSHPINSA'].rolling(window=30).std()
    
    #crude oil 
    # Step 2: Assign the EFFR column to the final DataFrame
    final['oil_WTI'] = oil['WTI Oil Price FOB (Dollars per Barrel)']

    final['oil_EURO'] = oil['Europe Brent Crude Oil (Dollars per Barrel)']

    # Step 3: Calculate the percentage change for EFFR
    final['oil_WTI_pctChange'] = oil['WTI Oil Price FOB (Dollars per Barrel)'].pct_change()

    final['oil_WTI_volt'] = oil['WTI Oil Price FOB (Dollars per Barrel)'].rolling(window=10).std()

    # Step 3: Calculate the percentage change for EFFR
    final['oil_EURO_pctChange'] = oil['Europe Brent Crude Oil (Dollars per Barrel)'].pct_change()

    final['oil_Euro_volt'] = oil['Europe Brent Crude Oil (Dollars per Barrel)'].rolling(window=10).std()
 
    # Fill missing values and prepare features for prediction
    final.fillna(0, inplace=True)
    X=final
    X = X.drop(columns=['date'])
    scaler = MinMaxScaler(feature_range=(-1, 1))
    X_scaled = scaler.fit_transform(X)  # X is your features DataFrame
    X_scaled_df = pd.DataFrame(X_scaled, columns=X.columns)
    dmatrix = xgb.DMatrix(X_scaled_df)
    
    
    

        # Load the pre-trained XGBoost model
    with open(os.path.join(input_path, 'pickles\\xgb_model_2.pkl'), 'rb') as model_file:
            xgb_model = pickle.load(model_file)

    # Make predictions
    predictions = xgb_model.predict(dmatrix)

    # Create output DataFrame and save to CSV
    output_df = pd.DataFrame({
        'date': final['date'],
        'prediction': predictions
    })

    output_df.to_csv(output_path, index=False)
    print(f"Predictions saved to {output_path}")



In [3]:
if __name__ == "__main__":
 # Argument parser to get input and output paths from command line
    parser = argparse.ArgumentParser(description="Process input and output file paths.")
    
    # Define input and output arguments
    parser.add_argument('--input_path', type=str, required=True, help='Path to the input directory.')
    parser.add_argument('--output_path', type=str, required=True, help='Path to the output CSV file.')
    
    # Parse the arguments
    args = parser.parse_args()
    
    # Call the main function with parsed arguments
    main(args.input_path, args.output_path)
  


usage: ipykernel_launcher.py [-h] --input_path INPUT_PATH --output_path OUTPUT_PATH
ipykernel_launcher.py: error: the following arguments are required: --input_path, --output_path


SystemExit: 2

C:\Users\youss\AppData\Local\Programs\Python\Python311\Lib\site-packages\IPython\core\interactiveshell.py:3516: UserWarning: To exit: use 'exit', 'quit', or Ctrl-D.
  warn("To exit: use 'exit', 'quit', or Ctrl-D.", stacklevel=1)
